# STEP 1-3. 데이터 수집 · 정제 · 교체 · 후처리

## 전체 흐름

```
[원본 parquet]
  상장기업.parquet  +  외감기업.parquet          → GAAP 통합본
  상장기업_IFRS.parquet + 외감기업_IFRS.parquet  → IFRS 통합본
        ↓ STEP 1
  기업_통합본.parquet          (GAAP 정제)
  기업_통합본_IFRS.parquet     (IFRS 정제)
        ↓ STEP 2-A
  기업_통합본_IFRS_정제.parquet  (IFRS 컬럼 → GAAP 체계로 재매핑)
        ↓ STEP 2-B
  기업_통합본_GAAP_IFRS교체.parquet  (GAAP 기업에 IFRS 값 주입)
        ↓ STEP 3
  기업_통합본_(todo).parquet   (3년 미만 제거 + 회사명 최신화)
```

## 실행 전 경로 확인
이 노트북은 `윤태/` 폴더 안에 있습니다.
모든 입출력 파일은 `윤태/데이터/` 폴더 하나에서 관리됩니다.
Jupyter를 `윤태/` 폴더에서 시작하면 아래 상대 경로가 자동으로 동작합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('데이터').resolve()

print(f'데이터 경로: {DATA_DIR}')

데이터 경로: C:\Users\ieoql\Documents\취업준비\code\Corporate Bankruptcy\corporate-bankruptcy\데이터 전처리\데이터


---
# STEP 1. 기업 원본 데이터 수집 및 정제

상장기업·외감기업 원본 parquet을 GAAP / IFRS 기준으로 각각 정제하고 통합합니다.

| 단계 | 내용 |
|------|------|
| STEP 1 | 결산월 12월 필터 (GAAP만, IFRS는 회계년도 컬럼에 포함) |
| STEP 2 | 유한회사 제거 — (유), （유）, 유한회사 텍스트 포함 기업 |
| STEP 3 | 외감기업에서 상장기업과 사업자번호 중복 제거 |
| STEP 4 | 동일 기업·연도 내부 중복 제거 |
| STEP 5 | 상장 + 외감 concat 통합 |
| STEP 6 | 금융·특수목적 기업 키워드 제거 (스팩, 유동화, 리츠 등) |

In [2]:
# ── 1-A. GAAP 원본 로드 ───────────────────────────────────────
df_listed = pd.read_parquet(DATA_DIR / '상장기업.parquet')
df_ext    = pd.read_parquet(DATA_DIR / '외감기업.parquet')
print(f'[원본 GAAP]  상장: {df_listed["사업자등록번호"].nunique():,}개  외감: {df_ext["사업자등록번호"].nunique():,}개')

[원본 GAAP]  상장: 2,453개  외감: 65,932개


In [3]:
# STEP 1 — 결산월 12월 필터
df_listed = df_listed[df_listed['결산월'] == 12].copy()
df_ext    = df_ext[df_ext['결산월'] == 12].copy()
print(f'[STEP 1] 결산월 12월  상장: {df_listed["사업자등록번호"].nunique():,}개  외감: {df_ext["사업자등록번호"].nunique():,}개')

# STEP 2 — 유한회사 제거
yuhan_pattern = r'\(유\)|（유）|유한회사'
df_listed = df_listed[~df_listed['회사명'].str.contains(yuhan_pattern, na=False)].copy()
df_ext    = df_ext[~df_ext['회사명'].str.contains(yuhan_pattern, na=False)].copy()
print(f'[STEP 2] 유한회사 제거  상장: {df_listed["사업자등록번호"].nunique():,}개  외감: {df_ext["사업자등록번호"].nunique():,}개')

# STEP 3 — 외감에서 상장 중복 제거
def normalize(series):
    return series.astype(str).str.replace('-', '', regex=False).str.strip()

listed_biz_set = set(normalize(df_listed['사업자등록번호'].dropna()))
df_ext = df_ext[~normalize(df_ext['사업자등록번호']).isin(listed_biz_set)].copy()
print(f'[STEP 3] 상장/외감 중복 제거  외감: {df_ext["사업자등록번호"].nunique():,}개')

# STEP 4 — 내부 중복 제거
key = ['사업자등록번호', '회계년도']
before_l, before_e = len(df_listed), len(df_ext)
df_listed = df_listed.drop_duplicates(subset=key, keep='first').reset_index(drop=True)
df_ext    = df_ext.drop_duplicates(subset=key, keep='first').reset_index(drop=True)
print(f'[STEP 4] 내부 중복  상장: -{before_l - len(df_listed):,}행  외감: -{before_e - len(df_ext):,}행')

# STEP 5 — 통합
df_listed['거래소코드'] = df_listed['거래소코드'].astype(str)
df_ext['거래소코드']    = df_ext['거래소코드'].astype(str)
df_gaap = pd.concat([df_listed, df_ext], ignore_index=True)
print(f'[STEP 5] 통합  {len(df_gaap):,}행  |  {df_gaap["사업자등록번호"].nunique():,}개')

# STEP 6 — 금융·특수목적 기업 제거
exclude_keywords = [
    '기업인수목적', '유동화', '자산유동화', '유동화전문',
    'ABS', 'PFV', '위탁관리부동산', '리츠', 'REITs',
    '선박투자', '부동산투자회사', '특수목적'
]
pattern = '|'.join(exclude_keywords)
before = df_gaap['사업자등록번호'].nunique()
df_gaap = df_gaap[~df_gaap['회사명'].str.contains(pattern, na=False, case=False)].copy()
print(f'[STEP 6] 금융특수목적 제거  -{before - df_gaap["사업자등록번호"].nunique():,}개  →  {df_gaap["사업자등록번호"].nunique():,}개')

print(f'\n남은 중복 관측치: {df_gaap.duplicated(subset=key, keep=False).sum():,}행')
print(f'결산월 12월 비율: {(df_gaap["결산월"] == 12).mean() * 100:.1f}%')

[STEP 1] 결산월 12월  상장: 2,453개  외감: 65,085개
[STEP 2] 유한회사 제거  상장: 2,451개  외감: 64,382개
[STEP 3] 상장/외감 중복 제거  외감: 63,480개
[STEP 4] 내부 중복  상장: -159행  외감: -6,014행
[STEP 5] 통합  438,930행  |  65,931개
[STEP 6] 금융특수목적 제거  -582개  →  65,349개

남은 중복 관측치: 0행
결산월 12월 비율: 100.0%


In [4]:
# ── 1-B. IFRS 원본 로드 및 정제 ──────────────────────────────
# IFRS는 결산월 필터 생략 (회계년도 컬럼이 '2010/12' 형식으로 이미 포함)
df_listed_i = pd.read_parquet(DATA_DIR / '상장기업_IFRS.parquet')
df_ext_i    = pd.read_parquet(DATA_DIR / '외감기업_IFRS.parquet')
print(f'[원본 IFRS]  상장: {df_listed_i["사업자등록번호"].nunique():,}개  외감: {df_ext_i["사업자등록번호"].nunique():,}개')

# 유한회사 / 상장-외감 중복 / 내부 중복 / 통합 / 금융특수목적 (GAAP과 동일 로직)
df_listed_i = df_listed_i[~df_listed_i['회사명'].str.contains(yuhan_pattern, na=False)].copy()
df_ext_i    = df_ext_i[~df_ext_i['회사명'].str.contains(yuhan_pattern, na=False)].copy()

listed_biz_i = set(normalize(df_listed_i['사업자등록번호'].dropna()))
df_ext_i = df_ext_i[~normalize(df_ext_i['사업자등록번호']).isin(listed_biz_i)].copy()

key_i = ['사업자등록번호', '회계년도']
df_listed_i = df_listed_i.drop_duplicates(subset=key_i, keep='first').reset_index(drop=True)
df_ext_i    = df_ext_i.drop_duplicates(subset=key_i, keep='first').reset_index(drop=True)

df_listed_i['거래소코드'] = df_listed_i['거래소코드'].astype(str)
df_ext_i['거래소코드']    = df_ext_i['거래소코드'].astype(str)
df_ifrs_raw = pd.concat([df_listed_i, df_ext_i], ignore_index=True)
df_ifrs_raw = df_ifrs_raw[~df_ifrs_raw['회사명'].str.contains(pattern, na=False, case=False)].copy()

print(f'[정제 완료]  {len(df_ifrs_raw):,}행  |  {df_ifrs_raw["사업자등록번호"].nunique():,}개')

[원본 IFRS]  상장: 2,453개  외감: 65,806개
[정제 완료]  440,095행  |  66,038개


---
# STEP 2. IFRS 컬럼 정리 및 GAAP→IFRS 교체

## 2-A. IFRS 컬럼 정리
IFRS 원시 컬럼명(예: `자산(*)(요약)(IFRS연결)(백만원)`)을 GAAP 컬럼명(예: `자산총계(요약)(백만원)`)으로 재매핑합니다.
- **연결 우선**: IFRS연결 컬럼에 값이 있으면 사용, NaN이면 IFRS별도 컬럼으로 보완
- **단위 변환**: 급여(급료)는 천원 → 백만원 (× 0.001)

## 2-B. GAAP→IFRS 값 교체
- **(사업자등록번호, 회계년도)** 키로 매칭하여 GAAP 값을 IFRS로 대체
- 교체 조건: `자산총계 > 0` — 0이거나 NaN이면 IFRS 데이터 불완전으로 판단, GAAP 유지
- IFRS 회계년도 `2010/12` 형식 → 12월분 선별 후 연도 정수(`2010`)로 변환

In [5]:
# ── IFRS 컬럼 매핑 테이블 ────────────────────────────────────
# (GAAP 컬럼명, IFRS연결 컬럼명, IFRS별도 컬럼명, 단위배수)
COL_MAP = [
    ('회사명',                                        '회사명',                                                            '회사명',                                                    1),
    ('회계년도',                                      '회계년도',                                                          '회계년도',                                                  1),
    ('종업원',                                        '종업원',                                                            '종업원',                                                    1),
    ('설립일',                                        '설립일',                                                            '설립일',                                                    1),
    ('사업자등록번호',                                 '사업자등록번호',                                                     '사업자등록번호',                                             1),
    ('금감원등록번호',                                 '금감원등록번호',                                                     '금감원등록번호',                                             1),
    ('외부감사기관',                                   '외부감사기관',                                                       '외부감사기관',                                               1),
    ('통계청 한국표준산업분류 코드 11차(대분류)',        '통계청 한국표준산업분류 코드 11차(대분류)',                           '통계청 한국표준산업분류 코드 11차(대분류)',                   1),
    ('통계청 한국표준산업분류 11차(중분류)',             '통계청 한국표준산업분류 11차(중분류)',                                '통계청 한국표준산업분류 11차(중분류)',                        1),
    ('자산총계(요약)(백만원)',                          '자산(*)(요약)(IFRS연결)(백만원)',                                    '자산(*)(요약)(IFRS)(백만원)',                                1),
    ('유동자산(요약)(백만원)',                          '유동자산(*)(요약)(IFRS연결)(백만원)',                                '유동자산(*)(요약)(IFRS)(백만원)',                            1),
    ('현금 및 현금성자산(요약)(백만원)',                '현금및현금성자산(요약)(IFRS연결)(백만원)',                            '현금및현금성자산(요약)(IFRS)(백만원)',                        1),
    ('매출채권(요약)(백만원)',                          '매출채권 및 기타유동채권(요약)(IFRS연결)(백만원)',                    '매출채권 및 기타유동채권(요약)(IFRS)(백만원)',                1),
    ('재고자산(요약)(백만원)',                          '재고자산(요약)(IFRS연결)(백만원)',                                   '재고자산(요약)(IFRS)(백만원)',                               1),
    ('비유동자산(요약)(백만원)',                        '비유동자산(*)(요약)(IFRS연결)(백만원)',                              '비유동자산(*)(요약)(IFRS)(백만원)',                          1),
    ('유형자산(요약)(백만원)',                          '유형자산(요약)(IFRS연결)(백만원)',                                   '유형자산(요약)(IFRS)(백만원)',                               1),
    ('부채총계(요약)(백만원)',                          '부채(*)(요약)(IFRS연결)(백만원)',                                    '부채(*)(요약)(IFRS)(백만원)',                                1),
    ('유동부채(요약)(백만원)',                          '유동부채(*)(요약)(IFRS연결)(백만원)',                                '유동부채(*)(요약)(IFRS)(백만원)',                            1),
    ('매입채무(요약)(백만원)',                          '매입채무 및 기타유동채무(요약)(IFRS연결)(백만원)',                    '매입채무 및 기타유동채무(요약)(IFRS)(백만원)',                1),
    ('단기차입금(요약)(백만원)',                        '단기차입금(요약)(IFRS연결)(백만원)',                                 '단기차입금(요약)(IFRS)(백만원)',                             1),
    ('유동성장기부채(요약)(백만원)',                    '유동성장기부채(요약)(IFRS연결)(백만원)',                             '유동성장기부채(요약)(IFRS)(백만원)',                         1),
    ('사채(요약)(백만원)',                              '사채(요약)(IFRS연결)(백만원)',                                       '사채(요약)(IFRS)(백만원)',                                   1),
    ('장기차입금(요약)(백만원)',                        '장기차입금(요약)(IFRS연결)(백만원)',                                 '장기차입금(요약)(IFRS)(백만원)',                             1),
    ('비유동부채(요약)(백만원)',                        '비유동부채 (*)(요약)(IFRS연결)(백만원)',                             '비유동부채 (*)(요약)(IFRS)(백만원)',                         1),
    ('자본총계(요약)(백만원)',                          '자본(*)(요약)(IFRS연결)(백만원)',                                    '자본(*)(요약)(IFRS)(백만원)',                                1),
    ('자본금(요약)(백만원)',                            '자본금(요약)(IFRS연결)(백만원)',                                     '자본금(요약)(IFRS)(백만원)',                                 1),
    ('자본잉여금(요약)(백만원)',                        '자본잉여금(요약)(IFRS연결)(백만원)',                                 '자본잉여금(요약)(IFRS)(백만원)',                             1),
    ('이익잉여금(요약)(백만원)',                        '이익잉여금(결손금)(요약)(IFRS연결)(백만원)',                          '이익잉여금(결손금)(요약)(IFRS)(백만원)',                      1),
    ('매출액(요약)(백만원)',                            '매출액(수익)(요약)(IFRS연결)(백만원)',                               '매출액(수익)(요약)(IFRS)(백만원)',                           1),
    ('매출원가(요약)(백만원)',                          '매출원가(요약)(IFRS연결)(백만원)',                                   '매출원가(요약)(IFRS)(백만원)',                               1),
    ('매출총이익(요약)(백만원)',                        '매출총이익(손실)(요약)(IFRS연결)(백만원)',                            '매출총이익(손실)(요약)(IFRS)(백만원)',                        1),
    ('판매비와 관리비(요약)(백만원)',                   '판매비와 관리비(물류원가 등 포함)(요약)(IFRS연결)(백만원)',           '판매비와 관리비(물류원가 등 포함)(요약)(IFRS)(백만원)',       1),
    ('급료',                                           '급여(IFRS연결)(천원)',                                               None,                                                        0.001),
    ('영업이익(요약)(백만원)',                          '* (정상)영업손익(보고서기재)(요약)(IFRS연결)(백만원)',               '* (정상)영업손익(보고서기재)(요약)(IFRS)(백만원)',           1),
    ('이자비용(요약)(백만원)',                          '*이자비용(요약)(IFRS연결)(백만원)',                                  '*이자비용(요약)(IFRS)(백만원)',                              1),
    ('당기순이익(요약)(백만원)',                        '당기순이익(손실)(요약)(IFRS연결)(백만원)',                            '당기순이익(손실)(요약)(IFRS)(백만원)',                        1),
    ('법인세비용차감전(계속사업)손익(요약)(백만원)',    '법인세비용차감전순이익(손실)(요약)(IFRS연결)(백만원)',               '법인세비용차감전순이익(손실)(요약)(IFRS)(백만원)',           1),
    ('(계속사업손익)법인세비용(요약)(백만원)',          '법인세비용(요약)(IFRS연결)(백만원)',                                 '법인세비용(요약)(IFRS)(백만원)',                             1),
    ('영업활동으로 인한 현금흐름(요약)(백만원)',        '영업활동으로 인한 현금흐름(간접법)(*)(요약)(IFRS연결)(백만원)',      '영업활동으로 인한 현금흐름(간접법)(*)(요약)(IFRS)(백만원)', 1),
    ('*감가상각비',                                    '*유무형자산 등 상각비(요약)(IFRS연결)(백만원)',                       '*유무형자산 등 상각비(요약)(IFRS)(백만원)',                  1),
]
print(f'매핑 컬럼 수: {len(COL_MAP)}개')

매핑 컬럼 수: 40개


In [6]:
# ── 2-A. IFRS 컬럼 정리 실행 ─────────────────────────────────
print(f'IFRS 원시: {len(df_ifrs_raw):,}행  |  {df_ifrs_raw["사업자등록번호"].nunique():,}개')

out = pd.DataFrame()
for gaap_name, ifrs_연결, ifrs_별도, factor in COL_MAP:
    if ifrs_연결 and ifrs_연결 in df_ifrs_raw.columns:
        col = df_ifrs_raw[ifrs_연결].copy()
        if ifrs_별도 and ifrs_별도 in df_ifrs_raw.columns:
            col = col.fillna(df_ifrs_raw[ifrs_별도])
    elif ifrs_별도 and ifrs_별도 in df_ifrs_raw.columns:
        col = df_ifrs_raw[ifrs_별도].copy()
    else:
        col = pd.Series(np.nan, index=df_ifrs_raw.index)
        print(f'  ※ 미발견: {gaap_name}')
    if factor != 1 and pd.api.types.is_numeric_dtype(col):
        col = col * factor
    out[gaap_name] = col

print(f'컬럼 정리 완료: {len(out.columns)}개 컬럼')

IFRS 원시: 440,095행  |  66,038개
컬럼 정리 완료: 40개 컬럼


In [7]:
# ── 2-B. GAAP→IFRS 교체 ──────────────────────────────────────
df_gaap = pd.read_parquet(DATA_DIR / '기업_통합본_정제.parquet')
df_ifrs = out.copy()
print(f'GAAP: {df_gaap["사업자등록번호"].nunique():,}개  IFRS: {df_ifrs["사업자등록번호"].nunique():,}개')

# 키 정규화
def norm_biz(s):
    return s.astype(str).str.replace('-', '', regex=False).str.strip()

def norm_year(s):
    return s.astype(str).str.split('/').str[0].str.strip().astype(int)

# IFRS 결산월 12월 필터 (2010/12 → 연도 2010으로 중복 없이 변환)
if df_ifrs['회계년도'].astype(str).str.contains('/').any():
    month = df_ifrs['회계년도'].astype(str).str.split('/').str[1]
    df_ifrs = df_ifrs[pd.to_numeric(month, errors='coerce') == 12].copy()
    print(f'IFRS 12월 필터 후: {df_ifrs["사업자등록번호"].nunique():,}개')

key = ['사업자등록번호', '회계년도']
for d in [df_gaap, df_ifrs]:
    d['사업자등록번호'] = norm_biz(d['사업자등록번호'])
    d['회계년도']      = norm_year(d['회계년도'])
df_gaap = df_gaap.drop_duplicates(subset=key, keep='first').reset_index(drop=True)
df_ifrs = df_ifrs.drop_duplicates(subset=key, keep='first').reset_index(drop=True)

GAAP: 65,349개  IFRS: 66,038개
IFRS 12월 필터 후: 65,940개


In [8]:
# 자산총계 > 0 인 IFRS 행만 유효 (0이나 NaN이면 GAAP 유지)
ANCHOR = '자산총계(요약)(백만원)'
nan_cnt  = df_ifrs[ANCHOR].isna().sum()
zero_cnt = (df_ifrs[ANCHOR] <= 0).sum()
print(f'IFRS 자산총계 NaN: {nan_cnt:,}행  |  0 이하: {zero_cnt:,}행  → 교체 제외')
df_ifrs_valid = df_ifrs[df_ifrs[ANCHOR].notna() & (df_ifrs[ANCHOR] > 0)].copy()
print(f'IFRS 유효 행: {len(df_ifrs_valid):,}행  |  {df_ifrs_valid["사업자등록번호"].nunique():,}개')

# 메타 컬럼 제외 후 교체 컬럼 결정
meta_cols = {
    '회사명', '회계년도', '종업원', '설립일', '사업자등록번호',
    '금감원등록번호', '외부감사기관',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 코드 11차(중분류)',
    '통계청 한국표준산업분류 11차(중분류)',
}
replace_cols = [c for c in df_gaap.columns if c in df_ifrs_valid.columns and c not in meta_cols]
print(f'교체 대상 재무 컬럼: {len(replace_cols)}개')

# 인덱스 기반 교체
df_result    = df_gaap.copy().set_index(key)
df_ifrs_idx  = df_ifrs_valid.set_index(key)
overlap_idx  = df_result.index.intersection(df_ifrs_idx.index)
print(f'교체 (기업, 회계년도) 쌍: {len(overlap_idx):,}개')
df_result.loc[overlap_idx, replace_cols] = df_ifrs_idx.loc[overlap_idx, replace_cols]
df_result = df_result.reset_index()
print(f'교체 완료: {df_result["사업자등록번호"].nunique():,}개 기업')

IFRS 자산총계 NaN: 382,426행  |  0 이하: 1,400행  → 교체 제외
IFRS 유효 행: 53,978행  |  7,887개
교체 대상 재무 컬럼: 31개
교체 (기업, 회계년도) 쌍: 53,702개
교체 완료: 65,349개 기업


---
# STEP 3. 기업 후처리

| 단계 | 내용 |
|------|------|
| STEP 1 | 회계년도 3년 미만 기업 제거 — ICR 3년 연속 기준 적용을 위해 최소 3개 연도 필요 |
| STEP 2 | 회사명 최신화 — 사업자등록번호 기준 가장 최근 연도의 회사명으로 전체 통일 |

In [9]:
df = df_result.copy()
print(f'원본: {len(df):,}행  |  유니크 기업: {df["사업자등록번호"].nunique():,}개')

# 기업별 보유 연도 분포 확인
year_dist = df.groupby('사업자등록번호')['회계년도'].nunique().value_counts().sort_index()
print('\n기업별 보유 회계년도 수 분포:')
print(year_dist.to_string())

원본: 435,169행  |  유니크 기업: 65,349개

기업별 보유 회계년도 수 분포:
회계년도
1      3624
2     11254
3      8071
4      7070
5      5027
6      4304
7      3098
8      2668
9      2341
10     2485
11     1464
12     1535
13     1332
14     1344
15     9732


In [10]:
# STEP 1 — 회계년도 3년 미만 기업 제거
year_count = df.groupby('사업자등록번호')['회계년도'].nunique()
valid_biz  = year_count[year_count >= 3].index
before = df['사업자등록번호'].nunique()
df = df[df['사업자등록번호'].isin(valid_biz)].copy()
print(f'[STEP 1] 3년 미만 제거  -{before - df["사업자등록번호"].nunique():,}개  →  {df["사업자등록번호"].nunique():,}개 기업  ({len(df):,}행)')

[STEP 1] 3년 미만 제거  -14,878개  →  50,471개 기업  (409,037행)


In [11]:
# STEP 2 — 회사명 최신화
latest_name = (
    df.sort_values('회계년도')
      .groupby('사업자등록번호')['회사명']
      .last()
)

# 사명 변경 이력 확인
changed = (
    df[['사업자등록번호', '회사명']]
    .drop_duplicates()
    .groupby('사업자등록번호')
    .filter(lambda x: len(x) > 1)
    .merge(latest_name.rename('최신_회사명'), on='사업자등록번호')
    .sort_values('사업자등록번호')
)
print(f'[STEP 2] 사명 변경 이력 기업: {changed["사업자등록번호"].nunique():,}개')
if len(changed) > 0:
    print(changed.to_string(index=False))

df['회사명'] = df['사업자등록번호'].map(latest_name)

[STEP 2] 사명 변경 이력 기업: 3개
   사업자등록번호        회사명     최신_회사명
2168110039  (주)제이콘텐트리  (주)콘텐트리중앙
2168110039  (주)콘텐트리중앙  (주)콘텐트리중앙
5068101452  (주)포스코케미칼  (주)포스코퓨처엠
5068101452  (주)포스코퓨처엠  (주)포스코퓨처엠
6218161477     (주)도스코 주식회사무진트레이딩
6218161477 주식회사무진트레이딩 주식회사무진트레이딩


In [12]:
print(f'최종: {len(df):,}행  |  유니크 기업: {df["사업자등록번호"].nunique():,}개')
print(f'회계년도 범위: {df["회계년도"].min()} ~ {df["회계년도"].max()}')

최종: 409,037행  |  유니크 기업: 50,471개
회계년도 범위: 2010 ~ 2024
